#  Exploración Inicial — Bases ELCA
### Proyecto: Modelamiento Predictivo de Transiciones Sociales en Colombia
*Curso:* Inteligencia Artificial con Aplicaciones en Economía I  
*Universidad Externado de Colombia*  
*Integrantes:* Camilo Rendón, Nicolás Cubillos, Talia Linares, Alejandro Velandia

---

La parte inicial tiene el propósito de: *cargar todas las bases de la ELCA desde Google Drive y explorar su estructura*.

Se inspecciona:
- Cuántas filas y columnas tiene cada base
- Qué tipos de datos contiene
- Cuántos valores nulos hay
- Cómo se llaman las columnas clave (especialmente el identificador llave)

In [ ]:
# Montar Google Drive en Colab
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive montado correctamente')

Mounted at /content/drive
Google Drive montado correctamente


## 2. Librerías

Se usan únicamente librerías estándar de análisis de datos en Python.  
pyreadstat es necesario para leer archivos .dta (formato Stata).

In [ ]:
# Instalación de librería para leer archivos Stata (.dta)
!pip install pyreadstat --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.7 MB/s eta 0:00:00


In [ ]:
# Configuración general de visualización
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


## 3. Configuración de rutas

Aquí se definen las rutas de cada archivo.

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
BASE_PATH = '/content/drive/MyDrive/ELCA_proyecto'

# Años disponibles
ONDAS = ['2010', '2013', '2016']

# Nombres de los módulos (urbano y rural)
MODULOS = ['UHogar', 'UPersonas', 'UGastos', 'UChoques',
           'RHogar', 'RPersonas', 'RGastos', 'RChoques']

# Extensión de los archivos
EXT = {
    '2010': '10.csv',
    '2013': '13.csv',
    '2016': '16.csv'
}
# Construir diccionario con todas las rutas
rutas = {}
for onda in ONDAS:
    rutas[onda] = {}
    for modulo in MODULOS:
        ruta = os.path.join(BASE_PATH, onda, modulo + EXT[onda])
        rutas[onda][modulo] = ruta

# Verificar qué archivos existen realmente
print('Verificando archivos en Drive...\n')
faltantes = []
for onda in ONDAS:
    for modulo in MODULOS:
        ruta = rutas[onda][modulo]
        existe = os.path.exists(ruta)
        estado = 'Bien' if existe else 'NO ENCONTRADO'
        print(f'  {estado}  {onda}/{modulo}{EXT}')
        if not existe:
            faltantes.append(f'{onda}/{modulo}{EXT}')

print(f'\n--- Resumen: {24 - len(faltantes)}/24 archivos encontrados ---')
if faltantes:
    print('\nArchivos no encontrados:')

Verificando archivos en Drive...

  Bien  2010/UHogar{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2010/UPersonas{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2010/UGastos{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2010/UChoques{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2010/RHogar{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2010/RPersonas{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2010/RGastos{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2010/RChoques{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2013/UHogar{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2013/UPersonas{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2013/UGastos{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2013/UChoques{'2010': '10.csv', '2013': '13.csv', '2016': '16.csv'}
  Bien  2013/RHogar{'2010': '10.csv', '2013': '13.csv', '201

## 4. Carga de todas las bases

Se carga cada archivo y se almacena en un diccionario con la estructura bases[onda][modulo].  
Por ejemplo: bases['2016']['UHogar'] devuelve el DataFrame de hogares urbanos 2016.

Si un archivo falla al cargar (por nombre incorrecto o formato), se registra el error sin detener la ejecución.

In [ ]:
def cargar_base(ruta):
    """Carga archivos detectando automáticamente el separador."""
    intentos = [
        {'sep': None, 'engine': 'python'},
        {'sep': '\t'},
        {'sep': ';'}
    ]
    for intento in intentos:
        try:
            return pd.read_csv(ruta, low_memory=False, **intento)
        except Exception:
            continue
    raise ValueError(f"No se pudo leer el archivo: {ruta}")

# ← Esta línea es la que faltaba
bases = {onda: {} for onda in ONDAS}
errores = []

print('Cargando bases...\n')

for onda in ONDAS:
    for modulo in MODULOS:
        ruta = rutas[onda][modulo]

        if os.path.exists(ruta):
            try:
                df = cargar_base(ruta)
                bases[onda][modulo] = df
                filas, cols = df.shape
                print(f'  {onda}/{modulo:<12} → {filas:>6} filas × {cols:>4} columnas')
            except Exception as e:
                errores.append((onda, modulo, str(e)))
                print(f'  {onda}/{modulo:<12} → ERROR REAL: {e}')
        else:
            print(f'  {onda}/{modulo:<12} → archivo no encontrado')

print(f'\n--- Carga completa. Errores reales: {len(errores)} ---')

Cargando bases...

  2010/UHogar       →   5275 filas ×  451 columnas
  2010/UPersonas    →  22179 filas ×  455 columnas
  2010/UGastos      → 184625 filas ×   10 columnas
  2010/UChoques     →   5275 filas ×   33 columnas
  2010/RHogar       →   4578 filas ×  384 columnas
  2010/RPersonas    →  21019 filas ×  415 columnas
  2010/RGastos      → 329616 filas ×   10 columnas
  2010/RChoques     →   4578 filas ×   25 columnas
  2013/UHogar       →   4910 filas ×  468 columnas
  2013/UPersonas    →  20574 filas ×  529 columnas
  2013/UGastos      → 353520 filas ×   14 columnas
  2013/UChoques     →  73650 filas ×   32 columnas
  2013/RHogar       →   4351 filas ×  522 columnas
  2013/RPersonas    →  19339 filas ×  723 columnas
  2013/RGastos      → 313272 filas ×   14 columnas
  2013/RChoques     →  73967 filas ×   32 columnas
  2016/UHogar       →   4860 filas ×  396 columnas
  2016/UPersonas    →  19298 filas ×  609 columnas
  2016/UGastos      → 349920 filas ×   17 columnas
  2016/UChoq

## 5. Inspección detallada de cada base

Para cada base cargada se muestra:
- *Shape:* número de filas y columnas
- *Columnas:* nombre de todas las variables disponibles
- *Tipos de datos:* qué tipo tiene cada columna (numérico, categórico, objeto)
- *Nulos:* cuántos valores faltantes hay por columna
- *Primeras filas:* muestra de los datos reales

In [ ]:
def inspeccionar_base(df, nombre):
    """Imprime un resumen completo de un DataFrame."""
    print('=' * 70)
    print(f'BASE: {nombre}')
    print('=' * 70)

    # Dimensiones
    print(f'\n  Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

    # Memoria aproximada
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f'  Memoria:     {mem_mb:.1f} MB')

    # Lista de columnas con tipo de dato
    print(f'\n  Columnas ({df.shape[1]} en total):')
    tipos = df.dtypes.reset_index()
    tipos.columns = ['columna', 'tipo']
    tipos['nulos'] = df.isnull().sum().values
    tipos['% nulos'] = (tipos['nulos'] / len(df) * 100).round(1)
    print(tipos.to_string(index=False))

    # Primeras 3 filas
    print(f'\n  Primeras 3 filas:')
    print(df.head(3).to_string())
    print()

# Ejecutar inspección para todas las bases cargadas
for onda in ONDAS:
    for modulo in MODULOS:
        if modulo in bases[onda]:
            inspeccionar_base(bases[onda][modulo], f'{onda} / {modulo}')

Se han truncado las últimas 5000 líneas del flujo de salida.
            cr_activ1_no  object  12862    61.20
           hor_aactv1_no float64  12863    61.20
           min_activ1_no float64  12863    61.20
            cr_activ2_no  object  12862    61.20
           hor_aactv2_no float64  13937    66.30
           min_activ2_no float64  13937    66.30
            cr_activ3_no  object  17727    84.30
           hor_aactv3_no float64  17793    84.70
           min_activ3_no float64  17793    84.70
            cr_activ4_no  object  17793    84.70
           hor_aactv4_no float64  20324    96.70
           min_activ4_no float64  20324    96.70
            cr_activ5_no  object  20324    96.70
           hor_aactv5_no float64  20928    99.60
           min_activ5_no float64  20928    99.60
            cr_activ6_no  object  20928    99.60
           hor_aactv6_no float64  21007    99.90
           min_activ6_no float64  21007    99.90
            estado_civil  object  12859    61.20
        

In [ ]:
def limpiar_y_filtrar_base(df, nombre, modulo, threshold_drop=0.9, threshold_warning=0.5):

    print(f'\n{"="*60}')
    print(f'PROCESANDO: {nombre}')
    print(f'{"="*60}')

    n = len(df)

    # -------------------------
    # 1. Porcentaje de nulos
    # -------------------------
    pct_nulos = df.isnull().sum() / n

    cols_drop = pct_nulos[pct_nulos > threshold_drop].index.tolist()
    cols_warning = pct_nulos[(pct_nulos > threshold_warning) & (pct_nulos <= threshold_drop)].index.tolist()

    # -------------------------
    # 2. Detectar IDs
    # -------------------------
    columnas_id = [col for col in df.columns if any(x in col.lower() for x in ['llave', 'hogar', 'id', 'folio', 'orden'])]

    # No eliminar IDs
    cols_drop = [col for col in cols_drop if col not in columnas_id]

    # -------------------------
    # 3. Eliminar por nulos
    # -------------------------
    df_clean = df.drop(columns=cols_drop)

    # -------------------------
    # 4. Reglas por módulo
    # -------------------------

    #  Hogares
    if 'hogar' in modulo.lower():
        extra_drop = [col for col in df_clean.columns if any(x in col.lower() for x in ['especifique', 'otro'])]
        df_clean = df_clean.drop(columns=extra_drop, errors='ignore')

    # Personas
    elif 'personas' in modulo.lower():
        extra_drop = [col for col in df_clean.columns if any(x in col.lower() for x in [
            'embarazo', 'gestacion', 'fecundidad'
        ])]
        df_clean = df_clean.drop(columns=extra_drop, errors='ignore')

    #  Choques
    elif 'choques' in modulo.lower():
        cols_keep = [col for col in df_clean.columns if any(x in col.lower() for x in [
            'choque', 'impacto', 'estrategia'
        ])]
        df_clean = df_clean[columnas_id + cols_keep]

    #  Gastos (NO filtrar fuerte aquí)
    elif 'gastos' in modulo.lower():
        pass  # se deja casi intacta para agregación después

    # -------------------------
    # 5. Reporte
    # -------------------------
    print(f' Eliminadas (>90% nulos): {len(cols_drop)}')
    print(f' Revisar (50%-90%): {len(cols_warning)}')
    print(f' Columnas finales: {df_clean.shape[1]}')

    return df_clean, cols_drop, cols_warning

In [ ]:
bases_limpias = {}

for onda in ONDAS:
    bases_limpias[onda] = {}

    for modulo in MODULOS:
        if modulo in bases[onda]:

            df = bases[onda][modulo]

            df_clean, drop, warning = limpiar_y_filtrar_base(
                df,
                nombre=f'{onda} / {modulo}',
                modulo=modulo
            )

            bases_limpias[onda][modulo] = df_clean


PROCESANDO: 2010 / UHogar
 Eliminadas (>90% nulos): 236
 Revisar (50%-90%): 68
 Columnas finales: 209

PROCESANDO: 2010 / UPersonas
 Eliminadas (>90% nulos): 200
 Revisar (50%-90%): 198
 Columnas finales: 255

PROCESANDO: 2010 / UGastos
 Eliminadas (>90% nulos): 1
 Revisar (50%-90%): 2
 Columnas finales: 9

PROCESANDO: 2010 / UChoques
 Eliminadas (>90% nulos): 28
 Revisar (50%-90%): 3
 Columnas finales: 1

PROCESANDO: 2010 / RHogar
 Eliminadas (>90% nulos): 190
 Revisar (50%-90%): 33
 Columnas finales: 187

PROCESANDO: 2010 / RPersonas
 Eliminadas (>90% nulos): 183
 Revisar (50%-90%): 175
 Columnas finales: 232

PROCESANDO: 2010 / RGastos
 Eliminadas (>90% nulos): 1
 Revisar (50%-90%): 2
 Columnas finales: 9

PROCESANDO: 2010 / RChoques
 Eliminadas (>90% nulos): 17
 Revisar (50%-90%): 6
 Columnas finales: 2

PROCESANDO: 2013 / UHogar
 Eliminadas (>90% nulos): 230
 Revisar (50%-90%): 69
 Columnas finales: 219

PROCESANDO: 2013 / UPersonas
 Eliminadas (>90% nulos): 238
 Revisar (50%-90%

In [ ]:
# =========================================
#  AJUSTE FINAL DE BASES LIMPIAS
# =========================================

bases_finales = {}
gastos_agregados = {}
choques_agregados = {}
personas_agregadas = {}

for onda in ONDAS:
    bases_finales[onda] = {}

    for modulo in MODULOS:
        if modulo in bases_limpias[onda]:

            df = bases_limpias[onda][modulo].copy()
            nombre = f"{onda} / {modulo}"

            print(f"\n{'='*60}")
            print(f"AJUSTE FINAL: {nombre}")
            print(f"{'='*60}")

            # =========================
            #  PERSONAS
            # =========================
            if 'personas' in modulo.lower():

                # eliminar columnas sin variación
                cols_low_var = [col for col in df.columns if df[col].nunique() <= 1]
                df = df.drop(columns=cols_low_var)

                # agregación por hogar
                cols_id = [col for col in df.columns if 'llave' in col.lower()]

                if cols_id:
                    llave = cols_id[0]

                    df_agg = df.groupby(llave).agg({
                        col: 'mean' for col in df.select_dtypes(include='number').columns if col != llave
                    }).reset_index()

                    personas_agregadas[onda] = df_agg

                bases_finales[onda][modulo] = df
                print(f" Personas procesadas: {df.shape}")

            # =========================
            # GASTOS
            # =========================
            elif 'gastos' in modulo.lower():

                cols_id = [col for col in df.columns if 'llave' in col.lower()]

                if cols_id:
                    llave = cols_id[0]

                    if 'valor' in df.columns:
                        df_agg = df.groupby(llave).agg({
                            'valor': ['sum', 'mean', 'count']
                        })

                        df_agg.columns = ['gasto_total', 'gasto_promedio', 'num_gastos']
                        df_agg = df_agg.reset_index()

                        gastos_agregados[onda] = df_agg

                bases_finales[onda][modulo] = df
                print(f"Gastos listos: {df.shape}")

            # =========================
            #  CHOQUES (corregido)
            # =========================
            elif 'choques' in modulo.lower():

                # indicador simple de choque
                df['tuvo_choque'] = df.notnull().any(axis=1).astype(int)

                cols_id = [col for col in df.columns if 'llave' in col.lower()]

                if cols_id:
                    llave = cols_id[0]

                    df_agg = df.groupby(llave)['tuvo_choque'].max().reset_index()
                    choques_agregados[onda] = df_agg

                bases_finales[onda][modulo] = df
                print(f" Choques ajustados: {df.shape}")

            # =========================
            #  HOGAR
            # =========================
            elif 'hogar' in modulo.lower():

                bases_finales[onda][modulo] = df
                print(f" Hogar listo: {df.shape}")

            else:
                bases_finales[onda][modulo] = df


# =========================================
# MERGE FINAL POR ONDA
# =========================================

# Llave correcta por onda (según los diccionarios de la ELCA)
LLAVE_POR_ONDA = {
    '2010': 'llave',
    '2013': 'llave',
    '2016': 'llave'
}

bases_modelo = {}

for onda in ONDAS:

    print(f"\n{'='*60}")
    print(f"MERGE FINAL: {onda}")
    print(f"{'='*60}")

    llave = LLAVE_POR_ONDA[onda]

    # ---------------------------
    # Unir urbano + rural en hogar
    # ---------------------------
    frames_hogar = []
    for modulo in ['UHogar', 'RHogar']:
        if modulo in bases_finales[onda]:
            df_h = bases_finales[onda][modulo].copy()
            df_h['zona'] = 'urbano' if modulo.startswith('U') else 'rural'
            frames_hogar.append(df_h)

    if not frames_hogar:
        print(f"  Sin bases de hogar para {onda}, se omite")
        continue

    hogar = pd.concat(frames_hogar, ignore_index=True)
    print(f"  Hogar (U+R): {hogar.shape}")

    # Verificar que la llave existe
    if llave not in hogar.columns:
        llaves_disponibles = [c for c in hogar.columns if 'llave' in c.lower()]
        print(f"  ADVERTENCIA: '{llave}' no encontrada. Disponibles: {llaves_disponibles}")
        if llaves_disponibles:
            llave = llaves_disponibles[0]
            print(f"  Usando: '{llave}'")
        else:
            print(f"  ERROR: No hay llave válida en {onda}")
            continue

    df_final = hogar.copy()

    # ---------------------------
    # Merge personas
    # ---------------------------
    if onda in personas_agregadas:
        llave_personas = llave if llave in personas_agregadas[onda].columns else None
        if llave_personas:
            df_final = df_final.merge(
                personas_agregadas[onda],
                on=llave_personas,
                how='left',
                suffixes=('', '_pers')
            )
            print(f"  + Personas merged: {df_final.shape}")
        else:
            print(f"  ADVERTENCIA: llave '{llave}' no está en personas_agregadas[{onda}]")

    # ---------------------------
    # Merge gastos
    # ---------------------------
    if onda in gastos_agregados:
        llave_gastos = llave if llave in gastos_agregados[onda].columns else None
        if llave_gastos:
            df_final = df_final.merge(
                gastos_agregados[onda],
                on=llave_gastos,
                how='left',
                suffixes=('', '_gast')
            )
            print(f"  + Gastos merged: {df_final.shape}")
        else:
            print(f"  ADVERTENCIA: llave '{llave}' no está en gastos_agregados[{onda}]")

    # ---------------------------
    # Merge choques
    # ---------------------------
    if onda in choques_agregados:
        llave_choques = llave if llave in choques_agregados[onda].columns else None
        if llave_choques:
            df_final = df_final.merge(
                choques_agregados[onda],
                on=llave_choques,
                how='left',
                suffixes=('', '_chq')
            )
            print(f"  + Choques merged: {df_final.shape}")
        else:
            print(f"  ADVERTENCIA: llave '{llave}' no está en choques_agregados[{onda}]")

    bases_modelo[onda] = df_final
    print(f"\n  BASE FINAL {onda}: {df_final.shape[0]} hogares × {df_final.shape[1]} columnas")


AJUSTE FINAL: 2010 / UHogar
 Hogar listo: (5275, 209)

AJUSTE FINAL: 2010 / UPersonas
 Personas procesadas: (22179, 251)

AJUSTE FINAL: 2010 / UGastos
Gastos listos: (184625, 9)

AJUSTE FINAL: 2010 / UChoques
 Choques ajustados: (5275, 2)

AJUSTE FINAL: 2010 / RHogar
 Hogar listo: (4578, 187)

AJUSTE FINAL: 2010 / RPersonas
 Personas procesadas: (21019, 227)

AJUSTE FINAL: 2010 / RGastos
Gastos listos: (329616, 9)

AJUSTE FINAL: 2010 / RChoques
 Choques ajustados: (4578, 3)

AJUSTE FINAL: 2013 / UHogar
 Hogar listo: (4910, 219)

AJUSTE FINAL: 2013 / UPersonas
 Personas procesadas: (20574, 283)

AJUSTE FINAL: 2013 / UGastos
Gastos listos: (353520, 11)

AJUSTE FINAL: 2013 / UChoques
 Choques ajustados: (73650, 4)

AJUSTE FINAL: 2013 / RHogar
 Hogar listo: (4351, 231)

AJUSTE FINAL: 2013 / RPersonas
 Personas procesadas: (19339, 291)

AJUSTE FINAL: 2013 / RGastos
Gastos listos: (313272, 11)

AJUSTE FINAL: 2013 / RChoques
 Choques ajustados: (73967, 4)

AJUSTE FINAL: 2016 / UHogar
 Hogar 

In [ ]:
# Diagnóstico 1: cómo se llama el identificador en 2010
print("=== COLUMNAS DE HOGAR 2010 ===")
for modulo in ['UHogar', 'RHogar']:
    if modulo in bases_finales['2010']:
        cols = bases_finales['2010'][modulo].columns.tolist()
        print(f"\n{modulo} — primeras 20 columnas:")
        print(cols[:20])

=== COLUMNAS DE HOGAR 2010 ===

UHogar — primeras 20 columnas:
['ola', 'zona', 'region', 'id_dpto', 'id_mpio', 'id_mpioU', 'consecutivo_c', 'estrato', 't_hogar', 'consecutivo', 't_personas', 'ordeninforma', 'tipo_vivienda', 'material_pisos', 'material_paredes', 'sp_energia', 'sp_estrato', 'sp_gasnatural', 'sp_acueducto', 'sp_alcantarillado']

RHogar — primeras 20 columnas:
['ola', 'zona', 'region', 'id_dpto', 'id_mpio', 'id_mpioU', 'consecutivo_c', 'consecutivo', 't_hogar', 't_personas', 'ordeninforma', 'tipo_vivienda', 'material_pisos', 'material_paredes', 'sp_energia', 'sp_gasnatural', 'sp_acueducto', 'sp_alcantarillado', 'sp_telefono', 'sp_recoleccion_basura']


In [ ]:
# Diagnóstico 2: verificar duplicados en personas_agregadas
for onda in ONDAS:
    if onda in personas_agregadas:
        df = personas_agregadas[onda]
        col_llave = df.columns[0]
        total = len(df)
        unicos = df[col_llave].nunique()
        print(f"{onda} — personas_agregadas: {total} filas, {unicos} llaves únicas, columna: '{col_llave}'")

2010 — personas_agregadas: 21019 filas, 21019 llaves únicas, columna: 'llave_ID_lb'
2013 — personas_agregadas: 16848 filas, 16848 llaves únicas, columna: 'llave_ID_lb'
2016 — personas_agregadas: 3815 filas, 3815 llaves únicas, columna: 'llave'


In [ ]:
# Diagnóstico 3: primeras columnas de personas y choques por onda
for onda in ONDAS:
    if onda in personas_agregadas:
        cols = personas_agregadas[onda].columns.tolist()
        print(f"\npersonas_agregadas {onda} — primeras 10 columnas:")
        print(cols[:10])

for onda in ONDAS:
    if onda in choques_agregados:
        cols = choques_agregados[onda].columns.tolist()
        print(f"\nchoques_agregados {onda} — primeras 5 columnas:")
        print(cols[:5])


personas_agregadas 2010 — primeras 10 columnas:
['llave_ID_lb', 'consecutivo', 'orden', 'nac_ano', 'edad', 'edad_meses', 'orden_pariente', 'orden_padre', 'orden_madre', 'id_dpto_nac']

personas_agregadas 2013 — primeras 10 columnas:
['llave_ID_lb', 'consecutivo', 'orden', 'llave', 'hogar', 'llaveper', 'id_dpto_nac', 'id_mpio_nac', 'nac_dia', 'nac_ano']

personas_agregadas 2016 — primeras 10 columnas:
['llave', 'consecutivo', 'hogar', 'llave_n16', 'hogar_n16', 'orden', 'llave_ID_lb', 'llaveper', 'llaveper_n16', 'edad']

choques_agregados 2013 — primeras 5 columnas:
['llave', 'tuvo_choque']

choques_agregados 2016 — primeras 5 columnas:
['llave', 'tuvo_choque']


In [ ]:
# Diagnóstico 4: primeras columnas de UHogar y RHogar 2013
for modulo in ['UHogar', 'RHogar']:
    if modulo in bases_finales['2013']:
        cols = bases_finales['2013'][modulo].columns.tolist()
        print(f"\n{modulo} 2013 — primeras 20 columnas:")
        print(cols[:20])


UHogar 2013 — primeras 20 columnas:
['ola', 'zona', 'zona_2010', 'region', 'RegionLb', 'id_dpto', 'id_mpio', 'id_mpioU', 'consecutivo_c', 'consecutivo', 'hogar', 'llave', 'estrato', 't_personas', 'tipo_vivienda', 'material_paredes', 'material_pisos', 'sp_energia', 'sp_gasnatural', 'sp_acueducto']

RHogar 2013 — primeras 20 columnas:
['ola', 'zona', 'zona_2010', 'region', 'RegionLb', 'id_dpto', 'id_mpio', 'id_mpioU', 'consecutivo_c', 'consecutivo', 'hogar', 'llave', 'proviene_2010', 't_personas', 'tipo_vivienda', 'material_paredes', 'material_pisos', 'sp_energia', 'sp_gasnatural', 'sp_acueducto']


In [ ]:
# ============================================================
# PASOS 1 y 2: Separar urbano/rural y apilar por onda
# La columna 'zona_fuente' indica el origen de cada hogar
# La llave varía por onda:
#   2010 → 'consecutivo'
#   2013 → 'llave'
#   2016 → 'llave'
# ============================================================

LLAVE_HOGAR = {
    '2010': 'consecutivo',
    '2013': 'llave',
    '2016': 'llave'
}

LLAVE_PERSONAS = {
    '2010': 'consecutivo',
    '2013': 'llave',
    '2016': 'llave'
}

bases_por_onda = {}

for onda in ONDAS:
    print(f"\n{'='*60}")
    print(f"ONDA {onda}")
    print(f"{'='*60}")

    llave_h = LLAVE_HOGAR[onda]
    llave_p = LLAVE_PERSONAS[onda]

    # --- Apilar UHogar + RHogar ---
    frames_hogar = []
    for modulo in ['UHogar', 'RHogar']:
        if modulo in bases_finales[onda]:
            df_h = bases_finales[onda][modulo].copy()
            df_h['zona_fuente'] = 'urbano' if modulo.startswith('U') else 'rural'
            frames_hogar.append(df_h)

    if not frames_hogar:
        print(f"  Sin bases de hogar — se omite")
        continue

    hogar = pd.concat(frames_hogar, ignore_index=True)
    print(f"  Hogar apilado (U+R): {hogar.shape}")

    # Verificar llave
    if llave_h not in hogar.columns:
        print(f"  ERROR: '{llave_h}' no encontrada en hogar {onda}")
        print(f"  Columnas disponibles: {hogar.columns.tolist()[:15]}")
        continue

    # Verificar duplicados en hogar
    dups = hogar[llave_h].duplicated().sum()
    if dups > 0:
        print(f"  ADVERTENCIA: {dups} llaves duplicadas en hogar — se conserva primera ocurrencia")
        hogar = hogar.drop_duplicates(subset=llave_h, keep='first')

    print(f"  Hogar final: {hogar.shape} | llave: '{llave_h}'")

    # --- Personas: ya están agregadas, solo verificar llave ---
    if onda in personas_agregadas:
        pers = personas_agregadas[onda].copy()
        if llave_p not in pers.columns:
            print(f"  ADVERTENCIA: '{llave_p}' no en personas — se omite personas")
            pers = None
        else:
            dups_p = pers[llave_p].duplicated().sum()
            if dups_p > 0:
                print(f"  ADVERTENCIA: {dups_p} duplicados en personas — se conserva primera ocurrencia")
                pers = pers.drop_duplicates(subset=llave_p, keep='first')
            print(f"  Personas: {pers.shape} | llave: '{llave_p}'")
    else:
        pers = None
        print(f"  Sin personas para {onda}")

    # --- Choques ---
    if onda in choques_agregados:
        chq = choques_agregados[onda].copy()
        llave_chq = 'llave' if 'llave' in chq.columns else None
        if llave_chq:
            print(f"  Choques: {chq.shape} | llave: '{llave_chq}'")
        else:
            chq = None
    else:
        chq = None
        print(f"  Sin choques para {onda}")

    bases_por_onda[onda] = {
        'hogar': hogar,
        'personas': pers,
        'choques': chq,
        'llave_h': llave_h,
        'llave_p': llave_p
    }

print("\nPaso 1 y 2 completos")


ONDA 2010
  Hogar apilado (U+R): (9853, 239)
  Hogar final: (9853, 239) | llave: 'consecutivo'
  ADVERTENCIA: 16441 duplicados en personas — se conserva primera ocurrencia
  Personas: (4578, 63) | llave: 'consecutivo'
  Sin choques para 2010

ONDA 2013
  Hogar apilado (U+R): (9261, 259)
  Hogar final: (9261, 259) | llave: 'llave'
  ADVERTENCIA: 12497 duplicados en personas — se conserva primera ocurrencia
  Personas: (4351, 68) | llave: 'llave'
  Choques: (4351, 2) | llave: 'llave'

ONDA 2016
  Hogar apilado (U+R): (8818, 240)
  ADVERTENCIA: 691 llaves duplicadas en hogar — se conserva primera ocurrencia
  Hogar final: (8127, 240) | llave: 'llave'
  Personas: (3815, 78) | llave: 'llave'
  Choques: (3778, 2) | llave: 'llave'

Paso 1 y 2 completos


In [ ]:
# ============================================================
# PASO 3: Merge de módulos dentro de cada onda
# Resultado: una fila por hogar con toda la información
# ============================================================

bases_onda_completa = {}

for onda in ONDAS:
    if onda not in bases_por_onda:
        print(f"  {onda} — no disponible, se omite")
        continue

    info    = bases_por_onda[onda]
    hogar   = info['hogar']
    pers    = info['personas']
    chq     = info['choques']
    llave_h = info['llave_h']
    llave_p = info['llave_p']

    print(f"\n{'='*60}")
    print(f"MERGE ONDA {onda}")
    print(f"{'='*60}")

    df = hogar.copy()

    # Merge personas
    if pers is not None:
        df = df.merge(pers, left_on=llave_h, right_on=llave_p,
                      how='left', suffixes=('', '_pers'))
        print(f"  + Personas: {df.shape}")

    # Merge choques
    if chq is not None:
        df = df.merge(chq, left_on=llave_h, right_on='llave',
                      how='left', suffixes=('', '_chq'))
        print(f"  + Choques:  {df.shape}")

    # Añadir columna de onda para identificar el año
    df['onda'] = int(onda)

    # Estandarizar nombre del identificador a 'id_hogar' en todas las ondas
    df = df.rename(columns={llave_h: 'id_hogar'})

    bases_onda_completa[onda] = df
    print(f"  BASE ONDA {onda}: {df.shape[0]} hogares × {df.shape[1]} columnas")

print("\n Paso 3 completo")


MERGE ONDA 2010
  + Personas: (9853, 301)
  BASE ONDA 2010: 9853 hogares × 302 columnas

MERGE ONDA 2013
  + Personas: (9261, 326)
  + Choques:  (9261, 327)
  BASE ONDA 2013: 9261 hogares × 328 columnas

MERGE ONDA 2016
  + Personas: (8127, 317)
  + Choques:  (8127, 318)
  BASE ONDA 2016: 8127 hogares × 319 columnas

 Paso 3 completo


In [ ]:
# ============================================================
# PASO 4: Unión longitudinal entre ondas y construcción
# de la variable dependiente de transición social
#
# Para construirla necesitamos una medida de pobreza en T y T+1
# Usamos el índice de riqueza (riqueza_pca) o en su defecto
# el ingreso total del hogar como proxy
#
# Categorías de transición:
#   0 = Estable no pobre   (no pobre en T y T+1)
#   1 = Vulnerable         (no pobre en T, pobre en T+1)
#   2 = Movilidad          (pobre en T, no pobre en T+1)
#   3 = Crónico            (pobre en T y T+1)
# ============================================================

# --- Identificar columna proxy de bienestar ---
# Buscamos en orden: riqueza_pca, ingreso estimado, gasto total
PROXY_BIENESTAR = None
for candidata in ['riqueza_pca', 'tercil2016', 'ing_trabnoagr',
                   'gasto_total', 'vr_gtos_mensuales']:
    for onda in ['2016', '2013', '2010']:
        if onda in bases_onda_completa:
            if candidata in bases_onda_completa[onda].columns:
                PROXY_BIENESTAR = candidata
                print(f"  Proxy de bienestar encontrada: '{PROXY_BIENESTAR}'")
                break
    if PROXY_BIENESTAR:
        break

if not PROXY_BIENESTAR:
    print("  ADVERTENCIA: No se encontró proxy de bienestar automática")
    print("  Columnas disponibles en 2016:")
    print(bases_onda_completa['2016'].columns.tolist()[:30])

  Proxy de bienestar encontrada: 'riqueza_pca'


In [ ]:
# --- Construir umbral de pobreza ---
# Se define como el percentil 40 del proxy en cada onda
# (umbral relativo dentro de la muestra)

umbrales = {}
for onda in ONDAS:
    if onda in bases_onda_completa and PROXY_BIENESTAR:
        col = bases_onda_completa[onda][PROXY_BIENESTAR].dropna()
        umbral = col.quantile(0.40)
        umbrales[onda] = umbral
        print(f"  Umbral pobreza {onda} (p40 de '{PROXY_BIENESTAR}'): {umbral:.2f}")

  Umbral pobreza 2010 (p40 de 'riqueza_pca'): -0.86
  Umbral pobreza 2013 (p40 de 'riqueza_pca'): -0.50
  Umbral pobreza 2016 (p40 de 'riqueza_pca'): -0.16


In [ ]:
# --- Construir variable de transición para cada par de ondas ---
# Par 1: 2010 → 2013
# Par 2: 2013 → 2016

def construir_transicion(df_t, df_t1, llave, proxy, umbral_t, umbral_t1, sufijo):
    """
    Une dos ondas por llave y construye la variable de transición.
    """
    cols_t  = ['id_hogar', proxy, 'zona_fuente', 'onda']
    cols_t1 = ['id_hogar', proxy]

    # Solo columnas que existen
    cols_t  = [c for c in cols_t  if c in df_t.columns]
    cols_t1 = [c for c in cols_t1 if c in df_t1.columns]

    merge = df_t[cols_t].merge(
        df_t1[cols_t1],
        on='id_hogar',
        how='inner',
        suffixes=(f'_t', f'_t1')
    )

    col_t  = f'{proxy}_t'
    col_t1 = f'{proxy}_t1'

    if col_t not in merge.columns:
        col_t = proxy
    if col_t1 not in merge.columns:
        col_t1 = proxy

    merge['pobre_t']  = (merge[col_t]  < umbral_t).astype(int)
    merge['pobre_t1'] = (merge[col_t1] < umbral_t1).astype(int)

    def clasificar(row):
        if   row['pobre_t'] == 0 and row['pobre_t1'] == 0: return 0  # Estable
        elif row['pobre_t'] == 0 and row['pobre_t1'] == 1: return 1  # Vulnerable
        elif row['pobre_t'] == 1 and row['pobre_t1'] == 0: return 2  # Movilidad
        else:                                               return 3  # Crónico

    merge['transicion'] = merge.apply(clasificar, axis=1)
    merge['par_ondas']  = sufijo

    return merge

# Par 2010 → 2013
if '2010' in bases_onda_completa and '2013' in bases_onda_completa and PROXY_BIENESTAR:
    trans_1013 = construir_transicion(
        bases_onda_completa['2010'],
        bases_onda_completa['2013'],
        llave='id_hogar',
        proxy=PROXY_BIENESTAR,
        umbral_t=umbrales.get('2010', 0),
        umbral_t1=umbrales.get('2013', 0),
        sufijo='2010_2013'
    )
    print(f"\nPar 2010→2013: {trans_1013.shape}")
    print(trans_1013['transicion'].value_counts().sort_index()
          .rename({0:'Estable', 1:'Vulnerable', 2:'Movilidad', 3:'Crónico'}))

# Par 2013 → 2016
if '2013' in bases_onda_completa and '2016' in bases_onda_completa and PROXY_BIENESTAR:
    trans_1316 = construir_transicion(
        bases_onda_completa['2013'],
        bases_onda_completa['2016'],
        llave='id_hogar',
        proxy=PROXY_BIENESTAR,
        umbral_t=umbrales.get('2013', 0),
        umbral_t1=umbrales.get('2016', 0),
        sufijo='2013_2016'
    )
    print(f"\nPar 2013→2016: {trans_1316.shape}")
    print(trans_1316['transicion'].value_counts().sort_index()
          .rename({0:'Estable', 1:'Vulnerable', 2:'Movilidad', 3:'Crónico'}))

print("\n Paso 4 completo")


Par 2010→2013: (0, 9)
Series([], Name: count, dtype: int64)

Par 2013→2016: (8060, 9)
transicion
Estable       3951
Vulnerable     800
Movilidad      886
Crónico       2423
Name: count, dtype: int64

 Paso 4 completo


In [ ]:
# ============================================================
# PASO 5: Construcción de la base final para modelamiento
#
# Se unen los dos pares de transición con la información
# completa de cada hogar en T (el momento de la predicción)
# Resultado: una fila por hogar-par con variables + etiqueta
# ============================================================

frames_finales = []

for onda_t, onda_t1, trans_df in [
    ('2010', '2013', trans_1013 if '2010' in bases_onda_completa else None),
    ('2013', '2016', trans_1316 if '2013' in bases_onda_completa else None)
]:
    if trans_df is None:
        continue

    # Unir con la base completa del momento T
    base_t = bases_onda_completa[onda_t].copy()

    df_modelo = trans_df[['id_hogar', 'transicion', 'par_ondas']].merge(
        base_t,
        on='id_hogar',
        how='inner'
    )

    frames_finales.append(df_modelo)
    print(f"  Par {onda_t}→{onda_t1}: {df_modelo.shape}")

# Apilar los dos pares
base_final = pd.concat(frames_finales, ignore_index=True)

# Mover 'transicion' al frente
cols = ['transicion', 'par_ondas', 'id_hogar'] + \
       [c for c in base_final.columns if c not in ['transicion', 'par_ondas', 'id_hogar']]
base_final = base_final[cols]

print(f"\n{'='*60}")
print(f"BASE FINAL DEL MODELO")
print(f"{'='*60}")
print(f"  Filas (hogares-par):  {base_final.shape[0]}")
print(f"  Columnas (variables): {base_final.shape[1]}")
print(f"\n  Distribución variable dependiente:")
print(base_final['transicion'].value_counts().sort_index()
      .rename({0:'0-Estable', 1:'1-Vulnerable', 2:'2-Movilidad', 3:'3-Crónico'}))
print(f"\n  Nulos por columna (top 10 con más nulos):")
nulos = (base_final.isnull().mean() * 100).sort_values(ascending=False)
print(nulos[nulos > 0].head(10).round(1))

print("\n Paso 5 completo — base lista")

  Par 2010→2013: (0, 304)
  Par 2013→2016: (8060, 330)

BASE FINAL DEL MODELO
  Filas (hogares-par):  8060
  Columnas (variables): 482

  Distribución variable dependiente:
transicion
0-Estable       3951
1-Vulnerable     800
2-Movilidad      886
3-Crónico       2423
Name: count, dtype: int64

  Nulos por columna (top 10 con más nulos):
uc_cagno_agua         100.00
uc_aeropuerto         100.00
uc_fabricas           100.00
uc_basureros          100.00
uc_plaza_mercado      100.00
uc_terminal_bus       100.00
prestamo_particular   100.00
cocina_es             100.00
uso_sanitario         100.00
sitio_alimentos       100.00
dtype: float64

 Paso 5 completo — base lista


/tmp/ipykernel_11567/2170220556.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  base_final = pd.concat(frames_finales, ignore_index=True)


In [ ]:
# Diagnóstico: verificar si consecutivo existe en 2013
# y si sus valores coinciden con los de 2010

print("Valores únicos de id_hogar en 2010 (muestra):")
print(sorted(bases_onda_completa['2010']['id_hogar'].dropna().unique())[:10])

print("\nValores únicos de id_hogar en 2013 (muestra):")
print(sorted(bases_onda_completa['2013']['id_hogar'].dropna().unique())[:10])

# Ver si consecutivo existe en 2013
if 'consecutivo' in bases_onda_completa['2013'].columns:
    print("\n 'consecutivo' existe en 2013")
    print("Valores (muestra):")
    print(sorted(bases_onda_completa['2013']['consecutivo'].dropna().unique())[:10])
else:
    print("\n 'consecutivo' NO existe en 2013")
    # Buscar columnas candidatas
    candidatas = [c for c in bases_onda_completa['2013'].columns
                  if any(x in c.lower() for x in ['consec', 'lb', '2010', 'base'])]
    print(f"Columnas candidatas: {candidatas}")

Valores únicos de id_hogar en 2010 (muestra):
[np.int64(111001), np.int64(111002), np.int64(111003), np.int64(111004), np.int64(111006), np.int64(111007), np.int64(111008), np.int64(111009), np.int64(111010), np.int64(111011)]

Valores únicos de id_hogar en 2013 (muestra):
[np.int64(11100101), np.int64(11100301), np.int64(11100305), np.int64(11100402), np.int64(11100601), np.int64(11100801), np.int64(11100802), np.int64(11100901), np.int64(11101001), np.int64(11101101)]

 'consecutivo' existe en 2013
Valores (muestra):
[np.int64(111001), np.int64(111003), np.int64(111004), np.int64(111006), np.int64(111008), np.int64(111009), np.int64(111010), np.int64(111011), np.int64(111012), np.int64(111014)]


In [ ]:
# Eliminar columnas con 100% de nulos en la base final
cols_antes = base_final.shape[1]
base_final = base_final.dropna(axis=1, how='all')
cols_despues = base_final.shape[1]

print(f"Columnas eliminadas (100% nulos): {cols_antes - cols_despues}")
print(f"Columnas restantes: {cols_despues}")
print(f"Shape final: {base_final.shape}")

Columnas eliminadas (100% nulos): 152
Columnas restantes: 330
Shape final: (8060, 330)


In [ ]:
# ============================================================
# CORRECCIÓN: Par 2010 → 2013
# El puente longitudinal es:
#   2010: id_hogar (renombrado desde consecutivo)
#   2013: consecutivo (que conserva el ID de línea base)
# ============================================================

if PROXY_BIENESTAR in bases_onda_completa['2010'].columns and \
   PROXY_BIENESTAR in bases_onda_completa['2013'].columns and \
   'consecutivo' in bases_onda_completa['2013'].columns:

    df_t  = bases_onda_completa['2010'].copy()
    df_t1 = bases_onda_completa['2013'].copy()

    # Seleccionar columnas necesarias de cada onda
    cols_t  = ['id_hogar', PROXY_BIENESTAR, 'zona_fuente', 'onda']
    cols_t1 = ['consecutivo', PROXY_BIENESTAR]

    cols_t  = [c for c in cols_t  if c in df_t.columns]
    cols_t1 = [c for c in cols_t1 if c in df_t1.columns]

    merge_1013 = df_t[cols_t].merge(
        df_t1[cols_t1],
        left_on='id_hogar',
        right_on='consecutivo',
        how='inner',
        suffixes=('_t', '_t1')
    )

    print(f"Hogares emparejados 2010→2013: {len(merge_1013)}")

    # Construir variable de transición
    col_t  = f'{PROXY_BIENESTAR}_t'
    col_t1 = f'{PROXY_BIENESTAR}_t1'

    umbral_t  = umbrales.get('2010', merge_1013[col_t].quantile(0.40))
    umbral_t1 = umbrales.get('2013', merge_1013[col_t1].quantile(0.40))

    merge_1013['pobre_t']  = (merge_1013[col_t]  < umbral_t).astype(int)
    merge_1013['pobre_t1'] = (merge_1013[col_t1] < umbral_t1).astype(int)

    def clasificar(row):
        if   row['pobre_t'] == 0 and row['pobre_t1'] == 0: return 0
        elif row['pobre_t'] == 0 and row['pobre_t1'] == 1: return 1
        elif row['pobre_t'] == 1 and row['pobre_t1'] == 0: return 2
        else:                                               return 3

    merge_1013['transicion'] = merge_1013.apply(clasificar, axis=1)
    merge_1013['par_ondas']  = '2010_2013'

    print(f"\nDistribución transición 2010→2013:")
    print(merge_1013['transicion'].value_counts().sort_index()
          .rename({0:'0-Estable', 1:'1-Vulnerable', 2:'2-Movilidad', 3:'3-Crónico'}))

else:
    print("ADVERTENCIA: No se pudo construir el par 2010→2013")
    print(f"  '{PROXY_BIENESTAR}' en 2010: {PROXY_BIENESTAR in bases_onda_completa['2010'].columns}")
    print(f"  '{PROXY_BIENESTAR}' en 2013: {PROXY_BIENESTAR in bases_onda_completa['2013'].columns}")
    print(f"  'consecutivo' en 2013: {'consecutivo' in bases_onda_completa['2013'].columns}")

Hogares emparejados 2010→2013: 9261

Distribución transición 2010→2013:
transicion
0-Estable       4376
1-Vulnerable    1019
2-Movilidad     1181
3-Crónico       2685
Name: count, dtype: int64


In [ ]:
# ============================================================
# RECONSTRUIR BASE FINAL incluyendo par 2010→2013
# ============================================================

frames_finales = []

# Par 2010 → 2013
if 'merge_1013' in dir() and len(merge_1013) > 0:
    base_t_2010 = bases_onda_completa['2010'].copy()
    df_par1 = merge_1013[['id_hogar', 'transicion', 'par_ondas']].merge(
        base_t_2010,
        on='id_hogar',
        how='inner'
    )
    frames_finales.append(df_par1)
    print(f"Par 2010→2013: {df_par1.shape}")

# Par 2013 → 2016
if 'trans_1316' in dir() and len(trans_1316) > 0:
    base_t_2013 = bases_onda_completa['2013'].copy()
    df_par2 = trans_1316[['id_hogar', 'transicion', 'par_ondas']].merge(
        base_t_2013,
        on='id_hogar',
        how='inner'
    )
    frames_finales.append(df_par2)
    print(f"Par 2013→2016: {df_par2.shape}")

# Apilar
base_final = pd.concat(frames_finales, ignore_index=True)

# Eliminar columnas 100% nulas
base_final = base_final.dropna(axis=1, how='all')

# Reordenar columnas
cols_frente = ['transicion', 'par_ondas', 'id_hogar']
cols_resto  = [c for c in base_final.columns if c not in cols_frente]
base_final  = base_final[cols_frente + cols_resto]

print(f"\n{'='*60}")
print(f"BASE FINAL COMPLETA")
print(f"{'='*60}")
print(f"  Total hogares:  {base_final.shape[0]}")
print(f"  Total columnas: {base_final.shape[1]}")
print(f"\n  Distribución variable dependiente:")
print(base_final['transicion'].value_counts().sort_index()
      .rename({0:'0-Estable', 1:'1-Vulnerable', 2:'2-Movilidad', 3:'3-Crónico'}))
print(f"\n  Por par de ondas:")
print(base_final['par_ondas'].value_counts())

Par 2010→2013: (9261, 304)
Par 2013→2016: (8060, 330)

BASE FINAL COMPLETA
  Total hogares:  17321
  Total columnas: 482

  Distribución variable dependiente:
transicion
0-Estable       8327
1-Vulnerable    1819
2-Movilidad     2067
3-Crónico       5108
Name: count, dtype: int64

  Por par de ondas:
par_ondas
2010_2013    9261
2013_2016    8060
Name: count, dtype: int64


In [ ]:
# ============================================================
# LIMPIEZA FINAL DE BASE_FINAL
# Antes de entrar al EDA
# ============================================================

print(f"Shape antes de limpieza: {base_final.shape}")

# 1. Eliminar columnas con más del 60% de nulos
threshold = 0.60
pct_nulos = base_final.isnull().mean()
cols_drop = pct_nulos[pct_nulos > threshold].index.tolist()

# Proteger columnas clave
cols_proteger = ['transicion', 'par_ondas', 'id_hogar']
cols_drop = [c for c in cols_drop if c not in cols_proteger]

base_final = base_final.drop(columns=cols_drop)
print(f"Columnas eliminadas (>{int(threshold*100)}% nulos): {len(cols_drop)}")

# 2. Eliminar columnas de texto libre (object con alta cardinalidad)
cols_texto = []
for col in base_final.select_dtypes(include='object').columns:
    if col in cols_proteger:
        continue
    if base_final[col].nunique() > 50:
        cols_texto.append(col)

base_final = base_final.drop(columns=cols_texto)
print(f"Columnas texto libre eliminadas: {len(cols_texto)}")

# 3. Eliminar columnas de identificadores redundantes
cols_id_redundantes = [c for c in base_final.columns
                       if any(x in c.lower() for x in
                              ['id_mpio', 'id_dpto', 'consecutivo_c',
                               'llave_n16', 'llaveper', 'hogar_n16'])
                       and c not in cols_proteger]
base_final = base_final.drop(columns=cols_id_redundantes, errors='ignore')
print(f"Identificadores redundantes eliminados: {len(cols_id_redundantes)}")

print(f"\nShape final limpio: {base_final.shape}")
print(f"\nTipos de datos:")
print(base_final.dtypes.value_counts())
print(f"\nNulos restantes (columnas con >0%):")
nulos_restantes = (base_final.isnull().mean() * 100).sort_values(ascending=False)
print(nulos_restantes[nulos_restantes > 0].head(15).round(1))

Shape antes de limpieza: (17321, 482)
Columnas eliminadas (>60% nulos): 296
Columnas texto libre eliminadas: 12
Identificadores redundantes eliminados: 4

Shape final limpio: (17321, 170)

Tipos de datos:
object     117
float64     27
int64       26
Name: count, dtype: int64

Nulos restantes (columnas con >0%):
grado_educ          57.50
fechai_ano_1        55.40
fechai_mes_1        55.40
religion1_2013      55.00
period_cuota_1      54.50
religion2_2013      54.30
fexhog_2013         53.50
fhog_2010           53.50
ayu_desastres_nat   53.50
equip_maq           53.50
act_ocasional       53.50
seguro_hogar        53.50
act_prestamo        53.50
fhog_2013           53.50
tercil2013          53.50
dtype: float64


In [ ]:
# ============================================================
# LIMPIEZA FINAL — PARTE 2
# El problema: 117 columnas object cuando deberían ser numéricas
# Muchas variables categóricas de la ELCA vienen como texto
# porque tienen códigos como '1.0', '2.0' o etiquetas Stata
# ============================================================

# 1. Convertir columnas object a numérico donde sea posible
cols_object = base_final.select_dtypes(include='object').columns.tolist()
cols_proteger = ['transicion', 'par_ondas', 'id_hogar', 'zona_fuente']

convertidas = []
no_convertidas = []

for col in cols_object:
    if col in cols_proteger:
        continue
    try:
        base_final[col] = pd.to_numeric(base_final[col], errors='raise')
        convertidas.append(col)
    except Exception:
        # Intentar con coerce (convierte errores a NaN)
        convertido = pd.to_numeric(base_final[col], errors='coerce')
        pct_nan_nuevo = convertido.isnull().mean()
        pct_nan_orig  = base_final[col].isnull().mean()
        # Solo convertir si no se pierde más del 10% adicional
        if (pct_nan_nuevo - pct_nan_orig) < 0.10:
            base_final[col] = convertido
            convertidas.append(col)
        else:
            no_convertidas.append(col)

print(f"Columnas convertidas a numérico: {len(convertidas)}")
print(f"Columnas que permanecen como object: {len(no_convertidas)}")
print(f"\nColumnas object restantes:")
print(no_convertidas)

Columnas convertidas a numérico: 0
Columnas que permanecen como object: 115

Columnas object restantes:
['zona', 'region', 'tipo_vivienda', 'material_pisos', 'material_paredes', 'sp_energia', 'sp_gasnatural', 'sp_acueducto', 'sp_alcantarillado', 'sp_telefono', 'sp_recoleccion_basura', 'eliminan_basura', 'uc_fabricas', 'uc_basureros', 'uc_plaza_mercado', 'uc_terminal_bus', 'uc_aeropuerto', 'uc_cagno_agua', 'uc_planta_tratamiento', 'uc_trans_hidrocarburo', 'uc_linea_alta_tension', 'inundacion', 'avalancha', 'creciente', 'hundimiento', 'terremoto', 'servicio_sanitario', 'uso_sanitario', 'obtencion_agua', 'sitio_alimentos', 'cocina_es', 'energia_cocinan', 'tenencia_vivienda', 'recursos_propios', 'credito_financiera', 'prestamo_familiar', 'otra_financiacion', 'religion', 'cambio_vivienda', 'n_television_cable', 'n_internet', 'familias_accion', 'jovenes_accion', 'sena', 'red_juntos', 'icbf', 'sub_desempleo', 'caja_subsprest', 'caja_saludrec', 'ayu_emergencias', 'prg_adultomayor', 'ayu_despla

In [ ]:
# ============================================================
# 2. Manejo de columnas object restantes
# Variables de religión → One Hot Encoding (nominal, sin orden)
# Resto de categóricas con pocos valores → Label Encoding
# Alta cardinalidad → eliminar
# ============================================================

from sklearn.preprocessing import LabelEncoder

cols_object_final = base_final.select_dtypes(include='object').columns.tolist()
cols_object_final = [c for c in cols_object_final if c not in cols_proteger]

# Identificar columnas de religión específicamente
cols_religion = [c for c in cols_object_final if 'religion' in c.lower()]
cols_resto     = [c for c in cols_object_final if c not in cols_religion]

cols_encode       = []
cols_eliminar_cat = []

for col in cols_resto:
    n_cats = base_final[col].nunique()
    if n_cats <= 20:
        cols_encode.append(col)
    else:
        cols_eliminar_cat.append(col)

# --- Eliminar alta cardinalidad ---
base_final = base_final.drop(columns=cols_eliminar_cat, errors='ignore')
print(f"Columnas alta cardinalidad eliminadas: {len(cols_eliminar_cat)}")
print(f"  → {cols_eliminar_cat}")

# --- One Hot Encoding para religión ---
print(f"\nColumnas de religión encontradas: {cols_religion}")
for col in cols_religion:
    n_cats = base_final[col].nunique()
    print(f"  {col} — {n_cats} categorías: {sorted(base_final[col].dropna().unique())}")

    dummies = pd.get_dummies(
        base_final[col],
        prefix=col,
        drop_first=True,
        dtype=int
    )
    base_final = pd.concat([
        base_final.drop(columns=[col]),
        dummies
    ], axis=1)
    print(f"  → Reemplazada por {dummies.shape[1]} columnas dummy")

# --- Label Encoding para resto de categóricas ---
le = LabelEncoder()
for col in cols_encode:
    mask = base_final[col].notna()
    if mask.sum() > 0:
        base_final.loc[mask, col] = le.fit_transform(
            base_final.loc[mask, col].astype(str)
        )
        base_final[col] = pd.to_numeric(base_final[col], errors='coerce')

print(f"\nColumnas con Label Encoding: {len(cols_encode)}")
print(f"  → {cols_encode}")
print(f"\nTipos de datos tras conversión:")
print(base_final.dtypes.value_counts())
print(f"\nShape: {base_final.shape}")

Columnas alta cardinalidad eliminadas: 0
  → []

Columnas de religión encontradas: []

Columnas con Label Encoding: 0
  → []

Tipos de datos tras conversión:
float64    103
int64       72
object       2
Name: count, dtype: int64

Shape: (16957, 177)


In [ ]:
# ============================================================
# 3. Imputación de nulos
# Estrategia: mediana por par_ondas
# La mediana es robusta a outliers y apropiada para
# variables categóricas ordinales y continuas sesgadas
# ============================================================

cols_imputar = base_final.select_dtypes(include=['float64', 'int64']).columns.tolist()
cols_imputar = [c for c in cols_imputar if c not in ['transicion', 'id_hogar']]

nulos_antes = base_final[cols_imputar].isnull().sum().sum()

for col in cols_imputar:
    if base_final[col].isnull().sum() > 0:
        # Imputar por mediana dentro de cada par de ondas
        base_final[col] = base_final.groupby('par_ondas')[col].transform(
            lambda x: x.fillna(x.median())
        )
        # Si aún quedan nulos (columna toda NaN en algún grupo), imputar global
        if base_final[col].isnull().sum() > 0:
            base_final[col] = base_final[col].fillna(base_final[col].median())

nulos_despues = base_final[cols_imputar].isnull().sum().sum()
print(f"Nulos antes de imputación: {nulos_antes:,}")
print(f"Nulos después de imputación: {nulos_despues:,}")
print(f"\nNulos totales en base_final: {base_final.isnull().sum().sum()}")

Nulos antes de imputación: 776,470
Nulos después de imputación: 0

Nulos totales en base_final: 0


In [ ]:
# ============================================================
# 4. Construcción de variables derivadas clave
# Estas variables no están directamente en la ELCA pero
# se pueden calcular y tienen alto poder predictivo
# ============================================================

# Hacinamiento: personas por cuarto para dormir
if 't_personas' in base_final.columns and 't_cuartos_dormir' in base_final.columns:
    base_final['hacinamiento'] = (
        base_final['t_personas'] / base_final['t_cuartos_dormir'].replace(0, np.nan)
    ).fillna(0)
    base_final['hacinamiento_critico'] = (base_final['hacinamiento'] > 3).astype(int)
    print(" Hacinamiento calculado")

# Índice de activos del hogar (suma de bienes)
cols_activos = [c for c in base_final.columns
                if c.startswith('n_') and c in base_final.columns]
if cols_activos:
    base_final['indice_activos'] = base_final[cols_activos].sum(axis=1)
    print(f" Índice de activos calculado ({len(cols_activos)} bienes)")

# Ingreso total del hogar
cols_ingreso = [c for c in base_final.columns
                if c.startswith('ing_') and c in base_final.columns]
if cols_ingreso:
    base_final['ingreso_total'] = base_final[cols_ingreso].sum(axis=1)
    print(f" Ingreso total calculado ({len(cols_ingreso)} fuentes)")

# Tasa de dependencia
if 't_personas' in base_final.columns and 'edad' in base_final.columns:
    # Proxy: personas totales como denominador, edad promedio como indicador
    base_final['tasa_dependencia_proxy'] = (
        base_final['t_personas'] / (base_final['edad'].replace(0, np.nan))
    ).fillna(0).clip(upper=5)
    print(" Tasa de dependencia proxy calculada")

# Privación en vivienda (IPM)
# 1 si tiene al menos una privación grave en vivienda
if all(c in base_final.columns for c in ['material_pisos', 'sp_acueducto', 'sp_alcantarillado']):
    base_final['privacion_vivienda'] = (
        (base_final['material_pisos'].isin([4, 5])).astype(int) +  # piso tierra/madera burda
        (base_final['sp_acueducto'] == 2).astype(int) +            # sin acueducto
        (base_final['sp_alcantarillado'] == 2).astype(int)         # sin alcantarillado
    ).clip(upper=1)
    print(" Privación vivienda calculada")

print(f"\nShape tras variables derivadas: {base_final.shape}")

 Hacinamiento calculado
 Índice de activos calculado (18 bienes)
 Tasa de dependencia proxy calculada
 Privación vivienda calculada

Shape tras variables derivadas: (17321, 172)


/tmp/ipykernel_11567/611665461.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_final['hacinamiento'] = (
/tmp/ipykernel_11567/611665461.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_final['hacinamiento_critico'] = (base_final['hacinamiento'] > 3).astype(int)
/tmp/ipykernel_11567/611665461.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.

In [ ]:
# ============================================================
# 5. VERIFICACIÓN FINAL
# ============================================================

print("=" * 60)
print("VERIFICACIÓN FINAL DE BASE_FINAL")
print("=" * 60)

print(f"\nDimensiones:        {base_final.shape[0]:,} filas × {base_final.shape[1]} columnas")
print(f"Nulos totales:      {base_final.isnull().sum().sum()}")
print(f"Duplicados:         {base_final.duplicated().sum()}")

print(f"\nTipos de datos:")
print(base_final.dtypes.value_counts())

print(f"\nVariable dependiente:")
dist = base_final['transicion'].value_counts().sort_index()
dist.index = ['0-Estable', '1-Vulnerable', '2-Movilidad', '3-Crónico']
for cat, n in dist.items():
    pct = n / len(base_final) * 100
    print(f"  {cat:<15} {n:>5} ({pct:.1f}%)")

print(f"\nPor par de ondas:")
print(base_final['par_ondas'].value_counts())

print(f"\nVariables disponibles (primeras 40):")
print(base_final.columns.tolist()[:40])

VERIFICACIÓN FINAL DE BASE_FINAL

Dimensiones:        17,321 filas × 172 columnas
Nulos totales:      0
Duplicados:         364

Tipos de datos:
float64    104
int64       66
object       2
Name: count, dtype: int64

Variable dependiente:
  0-Estable        8327 (48.1%)
  1-Vulnerable     1819 (10.5%)
  2-Movilidad      2067 (11.9%)
  3-Crónico        5108 (29.5%)

Por par de ondas:
par_ondas
2010_2013    9261
2013_2016    8060
Name: count, dtype: int64

Variables disponibles (primeras 40):
['transicion', 'par_ondas', 'id_hogar', 'ola', 'zona', 'region', 'estrato', 't_hogar', 't_personas', 'ordeninforma', 'tipo_vivienda', 'material_pisos', 'material_paredes', 'sp_energia', 'sp_estrato', 'sp_gasnatural', 'sp_acueducto', 'sp_alcantarillado', 'sp_telefono', 'sp_recoleccion_basura', 'eliminan_basura', 't_hogares', 'uc_fabricas', 'uc_basureros', 'uc_plaza_mercado', 'uc_terminal_bus', 'uc_aeropuerto', 'uc_cagno_agua', 'uc_planta_tratamiento', 'uc_trans_hidrocarburo', 'uc_linea_alta_tension',

In [ ]:
# ============================================================
# CONSTRUCCIÓN DEL IPM DESDE LAS VARIABLES ELCA
# Siguiendo la metodología oficial del DNP
# Un hogar es pobre si tiene privación en al menos
# 33% de los indicadores ponderados (5 dimensiones)
# ============================================================

# --- DIMENSIÓN 1: EDUCACIÓN ---
if 'grado_educ' in base_final.columns:
    # Bajo logro educativo: jefe sin básica primaria completa
    base_final['priv_educacion'] = (base_final['grado_educ'] < 3).astype(int)
    print(" Privación educación")

# --- DIMENSIÓN 2: NIÑEZ Y JUVENTUD ---
# Trabajo infantil proxy: no disponible directamente en base agregada
# Se deja como 0 si no hay variable disponible
base_final['priv_ninez'] = 0
print("  Privación niñez: no disponible en base agregada — se asigna 0")

# --- DIMENSIÓN 3: TRABAJO ---
# Empleo informal o desempleo de larga duración
# En RPersonas/UPersonas hay variables de empleo
cols_empleo = [c for c in base_final.columns
               if any(x in c.lower() for x in ['informal', 'desempleo', 'ocup'])]
print(f"  Columnas de empleo disponibles: {cols_empleo}")

# --- DIMENSIÓN 4: SALUD ---
if 'sp_acueducto' in base_final.columns:
    base_final['priv_salud'] = (base_final['sp_acueducto'] == 2).astype(int)
    print(" Privación salud (proxy: sin acueducto)")

# --- DIMENSIÓN 5: VIVIENDA ---
if 'privacion_vivienda' in base_final.columns:
    base_final['priv_vivienda'] = base_final['privacion_vivienda']
    print(" Privación vivienda (ya calculada)")

# --- IPM SINTÉTICO ---
dims_disponibles = [c for c in ['priv_educacion', 'priv_ninez',
                                  'priv_salud', 'priv_vivienda']
                    if c in base_final.columns]

if dims_disponibles:
    base_final['ipm_sintetico'] = base_final[dims_disponibles].mean(axis=1)
    base_final['pobre_ipm'] = (base_final['ipm_sintetico'] >= 0.33).astype(int)
    print(f"\n IPM sintético construido con {len(dims_disponibles)} dimensiones")
    print(f"   % hogares pobres por IPM: {base_final['pobre_ipm'].mean()*100:.1f}%")

 Privación educación
  Privación niñez: no disponible en base agregada — se asigna 0
  Columnas de empleo disponibles: ['sub_desempleo']
 Privación salud (proxy: sin acueducto)
 Privación vivienda (ya calculada)

 IPM sintético construido con 4 dimensiones
   % hogares pobres por IPM: 38.2%


/tmp/ipykernel_11567/591574789.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_final['priv_educacion'] = (base_final['grado_educ'] < 3).astype(int)
/tmp/ipykernel_11567/591574789.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_final['priv_ninez'] = 0
/tmp/ipykernel_11567/591574789.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a

In [ ]:
# ============================================================
# RECONSTRUIR TRANSICIÓN CON IPM
# Más riguroso que usar riqueza_pca como proxy
# ============================================================

# Esto requiere tener pobre_ipm en T y en T+1
# Como ya tienes la base apilada, puedes comparar
# el ipm_sintetico entre pares de ondas

# Por ahora verificar si la distribución cambia mucho
if 'pobre_ipm' in base_final.columns:
    print("Distribución pobreza IPM por par de ondas:")
    print(base_final.groupby('par_ondas')['pobre_ipm'].mean().round(3))

    print("\nCorrelación riqueza_pca vs IPM sintético:")
    if 'riqueza_pca' in base_final.columns:
        corr = base_final['riqueza_pca'].corr(base_final['ipm_sintetico'])
        print(f"  r = {corr:.3f}")

Distribución pobreza IPM por par de ondas:
par_ondas
2010_2013   0.01
2013_2016   0.81
Name: pobre_ipm, dtype: float64

Correlación riqueza_pca vs IPM sintético:
  r = 0.071


In [ ]:
# ============================================================
# ESTRUCTURA DE PANEL
# id_hogar × onda como índice jerárquico
# ============================================================

panel = base_final.set_index(['id_hogar', 'par_ondas']).sort_index()

print(f"Panel: {panel.shape}")
print(f"Hogares únicos: {base_final['id_hogar'].nunique()}")
print(f"Períodos: {base_final['par_ondas'].unique()}")

# Verificar cuántos hogares aparecen en ambos períodos
hogares_ambos = (base_final.groupby('id_hogar')['par_ondas']
                 .nunique()
                 .eq(2)
                 .sum())
print(f"Hogares en ambos períodos: {hogares_ambos}")
print(f"Hogares solo en un período: {len(base_final['id_hogar'].unique()) - hogares_ambos}")

Panel: (17321, 176)
Hogares únicos: 16789
Períodos: ['2010_2013' '2013_2016']
Hogares en ambos períodos: 0
Hogares solo en un período: 16789


In [ ]:
# ============================================================
# CHEQUEO FINAL COMPLETO
# ============================================================

print("=" * 60)
print("RESUMEN FINAL — BASE LISTA PARA MODELAMIENTO")
print("=" * 60)

print(f"\n Dimensiones:         {base_final.shape[0]:,} filas × {base_final.shape[1]} columnas")
print(f" Nulos totales:       {base_final.isnull().sum().sum()}")
print(f" Duplicados:          {base_final.duplicated().sum()}")

print(f"\n Variable dependiente (transicion):")
dist = base_final['transicion'].value_counts().sort_index()
etiquetas = {0:'Estable', 1:'Vulnerable', 2:'Movilidad', 3:'Crónico'}
for k, n in dist.items():
    print(f"   {k} - {etiquetas[k]:<12} {n:>5} ({n/len(base_final)*100:.1f}%)")

print(f"\n Por período:")
print(base_final['par_ondas'].value_counts().to_string())

print(f"\n  Por zona:")
if 'zona_fuente' in base_final.columns:
    print(base_final['zona_fuente'].value_counts().to_string())

print(f"\n Tipos de columnas:")
print(base_final.dtypes.value_counts().to_string())

print(f"\n Variables clave presentes:")
vars_clave = ['transicion', 'par_ondas', 'zona_fuente', 'riqueza_pca',
              'ipm_sintetico', 'pobre_ipm', 'hacinamiento', 'indice_activos',
              'ingreso_total', 'privacion_vivienda', 'material_pisos',
              'sp_acueducto', 'sp_alcantarillado', 'n_internet',
              'tenencia_vivienda', 't_personas']
for v in vars_clave:
    estado = "" if v in base_final.columns else "no"
    print(f"   {estado} {v}")

# Guardar versión final
ruta = '/content/drive/MyDrive/ELCA_proyecto/base_final_modelo.csv'
base_final.to_csv(ruta, index=False)
print(f"\n Guardada en: {ruta}")

RESUMEN FINAL — BASE LISTA PARA MODELAMIENTO

 Dimensiones:         17,321 filas × 178 columnas
 Nulos totales:       0
 Duplicados:          364

 Variable dependiente (transicion):
   0 - Estable       8327 (48.1%)
   1 - Vulnerable    1819 (10.5%)
   2 - Movilidad     2067 (11.9%)
   3 - Crónico       5108 (29.5%)

 Por período:
par_ondas
2010_2013    9261
2013_2016    8060

  Por zona:
zona_fuente
urbano    8799
rural     8522

 Tipos de columnas:
float64    105
int64       71
object       2

 Variables clave presentes:
    transicion
    par_ondas
    zona_fuente
    riqueza_pca
    ipm_sintetico
    pobre_ipm
    hacinamiento
    indice_activos
   no ingreso_total
    privacion_vivienda
    material_pisos
    sp_acueducto
    sp_alcantarillado
    n_internet
    tenencia_vivienda
    t_personas

 Guardada en: /content/drive/MyDrive/ELCA_proyecto/base_final_modelo.csv


In [ ]:
# Eliminar duplicados conservando primera ocurrencia
antes = len(base_final)
base_final = base_final.drop_duplicates(keep='first')
print(f"Duplicados eliminados: {antes - len(base_final)}")
print(f"Shape final: {base_final.shape}")

Duplicados eliminados: 364
Shape final: (16957, 178)


In [ ]:
# Buscar columnas de ingreso disponibles
cols_ing = [c for c in base_final.columns if 'ing_' in c.lower()]
print(f"Columnas de ingreso encontradas: {cols_ing}")

if cols_ing:
    base_final['ingreso_total'] = base_final[cols_ing].sum(axis=1)
    print(f" ingreso_total construido desde: {cols_ing}")
else:
    # Si no hay columnas ing_, buscar alternativas
    cols_alt = [c for c in base_final.columns
                if any(x in c.lower() for x in ['gasto', 'vr_gto', 'salario'])]
    print(f"Alternativas disponibles: {cols_alt}")

Columnas de ingreso encontradas: []
Alternativas disponibles: []


In [ ]:
# ============================================================
# TRATAMIENTO DE VARIABLES DE RELIGIÓN
# ============================================================

# 1. religion → renombrar como indicador de práctica religiosa
base_final = base_final.rename(columns={'religion': 'practica_religiosa'})
print(" 'religion' renombrada a 'practica_religiosa'")
print(f"   Valores: {sorted(base_final['practica_religiosa'].unique())}")

# 2. religion1_2013 → colapsar en binaria: católico vs no católico
#    porque el 90% es católico y las demás categorías son muy pequeñas
if 'religion1_2013' in base_final.columns:
    base_final['es_catolico'] = (base_final['religion1_2013'] == 1).astype(int)
    base_final = base_final.drop(columns=['religion1_2013'])
    print("\n 'religion1_2013' → 'es_catolico' (1=Católico, 0=Otro/NaN)")
    print(base_final['es_catolico'].value_counts())

# 3. religion2_2013 → eliminar
#    Es la segunda religión del hogar, casi todos tienen 5 (ninguna)
#    Agrega ruido sin valor predictivo real
if 'religion2_2013' in base_final.columns:
    base_final = base_final.drop(columns=['religion2_2013'])
    print("\n 'religion2_2013' eliminada (95% valor 5 = ninguna segunda religión)")

print(f"\nShape tras tratamiento religión: {base_final.shape}")

# ============================================================
# Guardar base actualizada en Drive
# ============================================================
ruta = '/content/drive/MyDrive/ELCA_proyecto/base_final_modelo.csv'
base_final.to_csv(ruta, index=False)
print(f"\nBase guardada: {base_final.shape[0]:,} hogares × {base_final.shape[1]} columnas")

 'religion' renombrada a 'practica_religiosa'
   Valores: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Shape tras tratamiento religión: (16957, 177)

Base guardada: 16,957 hogares × 177 columnas
